# A3.8 · Environment separation

**Function A — Security Architecture & Platform → The Platform & Cloud Security Engineer**  ·  *Security of AI*

---

**Risk.** "The agent knows not to" is not separation.

**Control.** Dev-agent credentials structurally unable to reach production.

**This lab.** Make dev-agent credentials structurally unable to reach production.

| | |
|---|---|
| Open-source tooling | SPIRE, OPA |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("A3.8"))

Environment separation for agents is not the same problem as for CI. An agent carries its context across boundaries, so the separation has to bind to the identity, not to the network.

In [ ]:
from cybercommons import identity, sandbox

ENVS = {
 "dev":     sandbox.Sandbox(
     egress=sandbox.EgressPolicy(allow_suffixes={".dev.internal"}),
     paths=sandbox.PathGuard(workspace="/work/dev"),
     tools=sandbox.ToolPolicy(allow={"read_file", "write_file", "run_shell"})),
 "prod":    sandbox.Sandbox(
     egress=sandbox.EgressPolicy(allow_hosts={"api.github.com"}),
     paths=sandbox.PathGuard(workspace="/work/prod"),
     tools=sandbox.ToolPolicy(allow={"read_file"},
                              require_approval={"write_file"},
                              deny={"run_shell"})),
}
for env, box in ENVS.items():
    print(env)
    for tool, target in [("read_file", f"/work/{env}/app.py"),
                         ("read_file", "/work/prod/secrets.yaml"),
                         ("run_shell", ""),
                         ("write_file", f"/work/{env}/out.txt")]:
        print("   ", box.call(tool, target))
    print()

The dev sandbox cannot reach `/work/prod` — not because of a network rule, but because its workspace is a different path and the guard normalises before checking. Separation that lives in the identity's policy travels with the agent; separation that lives in a VPC does not.

### Expect

Each environment permits reads inside its own workspace only. The dev box is denied the prod secrets path; the prod box denies `run_shell` outright and gates writes.

### Your turn

An agent debugging a prod incident needs prod read access from a dev context. Design that as a JIT grant rather than a second credential — `identity.JITGrant` is the shape.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/A3.8.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*